# Imports and Paths

In [1]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import hdf5plugin  # to read the compressed data
import os
import sys
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from tqdm.auto import tqdm
import stable_worldmodel as swm  # to load and run LeWM
from stable_worldmodel.data.formats.hdf5 import HDF5Dataset
import stable_pretraining as spt

/workspace/Safety-Dial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Paths and Helper Imports

In [2]:
here = Path.cwd().resolve()
REPO_ROOT = next(
    p for p in [here, *here.parents]
    if (p / "scripts" / "download_data.py").is_file()
)
# should print /workspace/Safety-Dial
print("REPO_ROOT", REPO_ROOT)

LEWM_DIR = REPO_ROOT / "third_party" / "le-wm"
if str(LEWM_DIR) not in sys.path:
    sys.path.insert(0, str(LEWM_DIR))

from utils import get_column_normalizer, get_img_preprocessor

H5_PATH = REPO_ROOT / "data" / "processed" / "pusht_expert_train.h5"
os.environ["STABLEWM_HOME"] = str(REPO_ROOT / "data" / "stablewm")  # assign, not setdefault

device = "cuda" if torch.cuda.is_available() else "cpu"
model = swm.policy.AutoCostModel("pusht/lewm").to(device).eval()

REPO_ROOT /workspace/Safety-Dial
16:17:38 | INFO  | __init__.py | JAX version 0.10.2 available.
16:17:41 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)


# Helper Funcs

In [3]:
# We use the same normalisation that they use in their pipeline during training
prep = get_img_preprocessor(source="pixels", target="pixels", img_size=224)

def pixels_to_tensor(frames_hwc):
    model_device = next(model.parameters()).device
    xs = []
    for f in frames_hwc:
        sample = {"pixels": f}
        prep(sample)
        xs.append(sample["pixels"])
    return torch.stack(xs, 0).unsqueeze(0).to(model_device)

#It get's super annoying when you open an already opened hd5 so
def closeh5():
    g = globals()
    f = g.get("file")
    if f is not None:
        f.close()

# Prepping the data

In [4]:
closeh5()

#Creating the dataset
dataset = HDF5Dataset(
    path=str(H5_PATH),
    num_steps=1, #One frame per sample
    frameskip=1, #Basically the temporal stride
    keys_to_load=["pixels", "state"], #We only need these to train the probe
    keys_to_cache=["state"], #This loads into RAM, pixels too big
    transform=prep)

MAX_SAMPLES = 50000 #Theres something like 2.3m 50k should suffice
#We sort because hdf5 reads faster when sequential
rng = np.random.default_rng(0)
n = len(dataset)
idx = rng.choice(n, size=min(MAX_SAMPLES, n), replace=False)
idx.sort()  # sequential HDF5 reads

#Splitting into train and test
n_train = int(0.8 * len(idx)) #40 000
train_idx, test_idx = idx[:n_train], idx[n_train:]
train_set = Subset(dataset, train_idx.tolist())
test_set = Subset(dataset, train_idx.tolist())

loader_kwargs = dict(batch_size=512, num_workers=0, pin_memory=True, shuffle=False) #512 should be good on a 16gb
train_loader = DataLoader(train_set, **loader_kwargs)
test_loader = DataLoader(test_set, **loader_kwargs)

# one batch: frame tensor + coords
batch = next(iter(train_loader))
pixels = batch["pixels"]                 # (B, 1, 3, 224, 224)
coords = batch["state"][:, 0, 2:4]       # (B, 2)  block_x, block_y
print(pixels.shape, pixels.dtype, coords.shape, coords.dtype)

16:17:41 | INFO  | __init__.py | Cached 'state' from '/workspace/Safety-Dial/data/processed/pusht_expert_train.h5'
torch.Size([512, 1, 3, 224, 224]) torch.float32 torch.Size([512, 2]) torch.float32


## Encoding both the loaders

In [5]:
@torch.inference_mode() #I think this disables grad tracking or something to that effect
def encode_loader(loader, desc="encode"):
    zs, ys = [], []
    for batch in tqdm(loader, desc=desc, total=len(loader)):
        pixels = batch["pixels"].to(device, non_blocking=True)
        y = batch["state"][:, 0, 2:4].float().numpy()  # block xy
        z = model.encode({"pixels": pixels})["emb"][:, 0].cpu().numpy()  # (B, 192) latent
        zs.append(z)
        ys.append(y)
    return np.concatenate(zs), np.concatenate(ys)

Z_train, Y_train = encode_loader(train_loader, desc="train")
Z_test, Y_test = encode_loader(test_loader, desc="val")  # or test_loader if that's what you named it
print(Z_train.shape, Y_train.shape, Z_test.shape, Y_test.shape)

train:  85%|████████▍ | 67/79 [07:18<01:18,  6.55s/it]


KeyboardInterrupt: 